# Chapter 4 §4.2~§4.4 실습 — 데이터 수집과 스키마 매핑

> 『AI 레디 데이터와 디지털 큐레이션』 Chapter 4의 실습 노트북입니다.
> 저장소: https://github.com/Suntae-Kim2020/digital-curation

## 학습 목표
- arXiv API에서 RAG 관련 논문 5건을 수집한다
- robots.txt를 점검하고 매너 있게 호출한다
- 수집 결과를 Ch.3 §3.4 스키마에 매핑해 JSON Lines로 저장한다
- Ch.3 §3.4.4 validate 함수로 필수 필드 누락을 점검한다

## 산출물
1. `ch04_collected.jsonl` — Ch.3 스키마에 매핑된 수집 결과

## 선행 학습
- Ch.3 §3.2 Dublin Core 15요소 / §3.3 AI 확장 6필드 / §3.4 스키마

## 0단계 · 환경 점검

Ch.3에서 pandas는 이미 설치했다. 본 챕터에서 새로 사용할 라이브러리는 세 가지다.

- **requests** — 사람을 위한 HTTP 라이브러리. 한 줄 GET 호출로 응답을 받는다.
- **beautifulsoup4** — HTML에서 원하는 요소(CSS 선택자)를 골라낸다.
- **lxml** — BeautifulSoup의 권장 파서 엔진. 깨진 HTML도 견고하게 처리하고 빠르다.

**설치 — VS Code 통합 터미널에서**

1. **단축키 Ctrl + 백틱(키보드 숫자 1 왼쪽 ~ 키)**을 누른다 — 또는 메뉴 Terminal → New Terminal.
2. 프롬프트 맨 앞에 `(.venv)` 표시가 보이는지 확인. 없다면 Ch.1 §1.7.4 절차로 .venv를 선택한다.
3. 다음 한 줄을 입력:

   ```
   pip install requests beautifulsoup4 lxml
   ```

4. 설치 확인:

   ```
   python -c "import requests, bs4, lxml; print('OK')"
   ```
   → `OK`가 나오면 성공. `.venv` 안에만 설치되어 시스템 Python에는 영향 없음.

In [ ]:
import requests, time, json
import pandas as pd
from xml.etree import ElementTree as ET
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

AGENT = 'AI-Curation-Course/1.0 (your-email@example.com)'
print('OK · requests', requests.__version__)

## 1단계 · arXiv API 호출 (인증키 불필요)

arXiv API는 키 없이 즉시 호출할 수 있고, RAG 관련 논문이 풍부해 학습용으로 최적이다.
응답은 Atom XML이다.

In [ ]:
BASE = 'https://export.arxiv.org/api/query'
params = {
    'search_query': 'all:retrieval augmented generation',
    'start': 0,
    'max_results': 5,
}
r = requests.get(BASE, params=params, headers={'User-Agent': AGENT}, timeout=15)
r.raise_for_status()

ns = {'atom': 'http://www.w3.org/2005/Atom'}
root = ET.fromstring(r.content)
entries = root.findall('atom:entry', ns)
print(f'받은 entry: {len(entries)}건')
for e in entries:
    title = e.find('atom:title', ns).text.strip()
    arxiv_id = e.find('atom:id', ns).text.split('/')[-1]
    print(f'  {arxiv_id} | {title[:60]}')

# arXiv 매너 — 다음 호출 전 3초
time.sleep(3)

## 2단계 · Ch.3 §3.4 스키마로 매핑

출처마다 필드 이름이 다르므로, 본서 표준 스키마(DC 15 + AI 6)에 맞춰 통일한다.
이 단계에서 의미 차원 필드(summary, keywords, chunk_ids)는 비워 두고 embedding_flag는 False로 두며
후속 챕터에서 단계적으로 채운다.

In [ ]:
def map_arxiv_entry(e, ns):
    arxiv_id = e.find('atom:id', ns).text.split('/')[-1]
    authors = [a.find('atom:name', ns).text
               for a in e.findall('atom:author', ns)]
    return {
        'id':           f'arxiv:{arxiv_id}',
        'title':        e.find('atom:title', ns).text.strip(),
        'creator':      '; '.join(authors),
        'subject':      [],
        'description':  e.find('atom:summary', ns).text.strip(),
        'publisher':    'arXiv',
        'contributor':  '',
        'date':         e.find('atom:published', ns).text[:10],
        'type':         'Article',
        'format':       'application/pdf',
        'identifier':   arxiv_id,
        'source':       '',
        'language':     'en',
        'relation':     '',
        'coverage':     '',
        'rights':       'arXiv non-exclusive license',
        'summary':      '',          # Ch.7 §7.4
        'keywords':     [],          # Ch.5 §5.3
        'source_url':   e.find('atom:id', ns).text,
        'license_code': 'ARXIV-NONEXCLUSIVE',
        'chunk_ids':    [],          # Ch.5 §5.4
        'embedding_flag': False,     # Ch.6 §6.5에서 True로
    }

records = [map_arxiv_entry(e, ns) for e in entries]
df = pd.DataFrame(records)
df[['id','date','language','license_code']]

## 3단계 · JSON Lines로 저장

리스트 필드(`subject`, `keywords`, `chunk_ids`)를 보존하기 위해 `.jsonl` 권장.
한글은 `force_ascii=False`로 그대로 둔다.

In [ ]:
df.to_json('ch04_collected.jsonl',
          orient='records', lines=True, force_ascii=False, date_format='iso')
print('Saved: ch04_collected.jsonl', '·', len(df), 'records')

with open('ch04_collected.jsonl', 'r', encoding='utf-8') as f:
    print('\n[첫 줄 미리보기]')
    print(f.readline()[:200], '…')

## 4단계 · robots.txt 점검 (웹 크롤링용)

API가 아닌 일반 페이지를 수집할 때는 매번 robots.txt를 점검한다.
법적 효력은 없지만 매너이자 차단 방지책이다.

In [ ]:
def can_collect(url, agent=AGENT):
    rp = RobotFileParser()
    # 도메인 루트의 robots.txt 위치 계산
    parts = url.split('/')
    rp.set_url(f'{parts[0]}//{parts[2]}/robots.txt')
    rp.read()
    allowed = rp.can_fetch(agent, url)
    delay   = rp.crawl_delay(agent) or 1
    return allowed, delay

for u in [
    'https://www.dublincore.org/specifications/',
    'https://schema.org/Book',
]:
    allowed, delay = can_collect(u)
    flag = '허용' if allowed else '금지'
    print(f'{flag} · 간격 {delay}초 · {u}')

## 5단계 · 정적 페이지에서 메타 추출 (예시)

robots.txt 통과 → HTML 수집 → BeautifulSoup으로 메타 추출의 표준 패턴.

In [ ]:
def fetch_meta(url, agent=AGENT):
    allowed, delay = can_collect(url, agent)
    if not allowed:
        return None
    r = requests.get(url, headers={'User-Agent': agent}, timeout=10)
    r.raise_for_status()
    s = BeautifulSoup(r.content, 'html.parser')
    title = s.title.text.strip() if s.title else ''
    desc = ''
    md = s.find('meta', attrs={'name': 'description'})
    if md and md.get('content'):
        desc = md['content'].strip()
    time.sleep(delay)
    return {'title': title, 'description': desc, 'source_url': url}

fetch_meta('https://www.dublincore.org/')

## 6단계 · Ch.3 §3.4.4 검증 함수 재사용

수집 결과의 필수 필드 누락 여부를 점검한다.

In [ ]:
def validate(records, schema):
    errors = []
    required = schema[schema['required']]['field'].tolist()
    for f in required:
        if f not in records.columns:
            errors.append(f'필수 필드 누락: {f}')
        else:
            nulls = records[records[f].isna() | (records[f] == '')].index.tolist()
            if nulls:
                errors.append(f'{f}: 값 누락 행 {nulls}')
    return errors

# Ch.3 스키마 불러오기 (저장소에 동봉)
import urllib.request
SCHEMA_URL = ('https://raw.githubusercontent.com/Suntae-Kim2020/'
              'digital-curation/main/ch03/ch03_schema.csv')
schema = pd.read_csv(SCHEMA_URL, encoding='utf-8-sig')
errors = validate(df, schema)
print('검증:', '통과' if not errors else errors)

## 다음 단계

- Ch.5 §5.1 : 수집한 description 본문을 정제
- Ch.5 §5.3 : 본문에서 keywords 자동 추출 (Kiwi)
- Ch.5 §5.4 : 청킹 → chunk_ids 채우기
- Ch.6 §6.5 : 벡터DB 적재 → embedding_flag True로 켜기
- Ch.7 §7.4 : LLM(Gemini)으로 summary 자동 생성